### Required Files/Directories

- **graph_credentials.json -** This file should be present under _`datahub-fork\datahub-integrations-service\experiments\docs_generation/`_ directory. It contains tokens for all instances
- **resource_files -** This directory needs to be present in _`datahub-fork\datahub-integrations-service\experiments\term_suggestion_v2\`_, it contains following files
  - **test_urns.json** - This file contains table urns for all instances
  - **.env** - Specify values of environment variables BEDROCK_AWS_ACCESS_KEY_ID and BEDROCK_AWS_SECRET_ACCESS_KEY
  - **care_columns_ground_truth.csv** - Ground truth CSV for Care URNs
  - **column_labels_without_care_data_final** - Ground truth CSV for Non-Care URNs
  - **care_glossary_terms_with_term_descriptions_and_examples.pkl -** Contains Care terms descriptions generated externally. This file can be generated using _"populate_glossary_terms_examples_and_descriptions.ipynb"_
  - **CARE_COLUMNS_INFO_WITH_AUTO_GENERATED_DESCRIPTIONS.PKL -** This file contains description for CARE instance columns generated using internal API. This file can be generated using _generate_descriptions_for_columns.ipynb_
  - **CARE_TABLES_INFO_WITH_AUTO_GENERATED_DESCRIPTIONS.PKL -** This file contains description for CARE instance tables generated using internal API. This file can be generated using _generate_descriptions_for_columns.ipynb_
- **prompts -** This directory is intended to contain all experimental prompts in a text file. This directory can be placed in _`datahub-fork\datahub-integrations-service\experiments\term_suggestion_v2\`_. This directory is checked in to Git.


In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
import os
import pathlib
import dotenv
import pickle
import sys
import json5
import pandas as pd
import numpy as np
from typing import Dict, List
from datetime import datetime

current_dir = pathlib.Path().parent.resolve()
sys.path.append(str(current_dir.parent))

from loguru import logger

# from datahub_integrations.gen_ai.bedrock import BedrockModel
from datahub_integrations.gen_ai.term_suggestion_v2 import get_term_recommendations
from datahub_integrations.gen_ai.term_suggestion_v2_context import (
    fetch_glossary_info,
    GlossaryUniverseConfig,
)

from docs_generation.graph_helper import create_datahub_graph
from helper import write_llm_output_to_csv

from term_suggestion_analysis_helper import (
    get_table_and_column_infos_dict,
    get_parsed_responses_for_single_experiment_run,
    populate_analysis_for_confidence_threshold_list,
    convert_parsed_response_to_readable_csv,
)


dotenv.load_dotenv(current_dir / "resource_files/.env")

In [29]:
from dataclasses import dataclass
from typing import List


@dataclass
class ExperimentConfig:
    prompt_path: str
    experiment_name: str
    iterations: int
    confidence_thresholds: List[float]


experiment_dict = [
    ExperimentConfig(
        prompt_path="prompts/FN_examples_reformatted.txt",
        experiment_name="FN_reformatted_examples_term_desc",
        iterations=3,
        confidence_thresholds=[7, 8, 8.5, 9, 9.5, 10],
    ),
    ExperimentConfig(
        prompt_path="prompts/FN_examples_user_id_inst.txt",
        experiment_name="FN_examples_term_desc_user_id_inst",
        iterations=3,
        confidence_thresholds=[7, 8, 8.5, 9, 9.5, 10],
    ),
]

In [ ]:
# Experiment settings:
EXAMPLES_IN_GLOSSARY_TERMS = False
USE_TERM_DESCRIPTIONS = True
USE_TERM_PARENT_INFO = True

USE_COLUMN_INFO_WITH_DESCRIPTIONS = False
OLD_LABELS = False

BASE_DIRECTORY = (
    current_dir / f"output/output_{datetime.now().strftime('%m-%d-%H-%M')}/"
)
print(BASE_DIRECTORY)
RESOURCE_DIR = current_dir / "resource_files/"
COLUMN_LABELS_CSV_PATH = RESOURCE_DIR / "care_columns_ground_truth.csv"
GLOSSARY_TERMS_INFO_FILE_PATH = (
    RESOURCE_DIR / "care_glossary_terms_with_term_descriptions_and_examples.pkl"
)

URNS_DICT_PATH = RESOURCE_DIR / "test_urns.json"
INSTANCES_TO_EXAMINE = ["care"]
GLOSSARY_INSTANCE = "care"
GLOSSARY_NODES = [
    "urn:li:glossaryNode:d966ad51-49a7-411c-8d06-2ee2dd210647",
    "urn:li:glossaryNode:09effdec-8810-415c-ace7-4e53090c502c",
]
GLOSSARY_TERMS = []


if OLD_LABELS:
    LABEL_COLUMN = "glossary_terms_updated"
else:
    LABEL_COLUMN = "glossary_terms_updated_new"

URNS_DICT: Dict[str, List[str]] = json5.loads(  # type: ignore
    URNS_DICT_PATH.read_text()
)


FILTERS = [
    "no_filter",
    # "description",
    # "sample_values",
    # "description_and_sample_values",
    # "description_or_sample_values",
    # "name_and_datatype",
]
URNS_DICT = {
    key: value for key, value in URNS_DICT.items() if key in INSTANCES_TO_EXAMINE
}

In [ ]:
if USE_COLUMN_INFO_WITH_DESCRIPTIONS:
    TABLE_INFO_FILE_PATH = (
        RESOURCE_DIR / "CARE_TABLES_INFO_WITH_AUTO_GENERATED_DESCRIPTIONS.PKL"
    )
    COLUMN_INFO_FILE_PATH = (
        RESOURCE_DIR / "CARE_COLUMNS_INFO_WITH_AUTO_GENERATED_DESCRIPTIONS.PKL"
    )
    with open(TABLE_INFO_FILE_PATH, "rb") as f:
        tables_info_dict = pickle.load(f)
        f.close()
    with open(COLUMN_INFO_FILE_PATH, "rb") as f:
        columns_info_dict = pickle.load(f)
        f.close()
else:
    tables_info_dict, columns_info_dict = get_table_and_column_infos_dict(
        urns_dict=URNS_DICT
    )

# Fetch Glossary terms


In [ ]:
if not EXAMPLES_IN_GLOSSARY_TERMS and not USE_TERM_DESCRIPTIONS:
    print("fetching terms......")
    graph_client = create_datahub_graph(GLOSSARY_INSTANCE)
    glossary_config = GlossaryUniverseConfig(
        glossary_nodes=GLOSSARY_NODES, glossary_terms=GLOSSARY_TERMS
    )
    glossary_info = fetch_glossary_info(
        graph_client=graph_client, universe=glossary_config
    )
else:
    print("loading terms......")
    with open(GLOSSARY_TERMS_INFO_FILE_PATH, "rb") as f:
        glossary_info = pickle.load(f)

if not EXAMPLES_IN_GLOSSARY_TERMS:
    print("removing examples......")
    for key, value in glossary_info.glossary.items():
        glossary_info.glossary[key].pop("example", None)

if not USE_TERM_PARENT_INFO:
    print("removing parent node info ......")
    for key, value in glossary_info.glossary.items():
        glossary_info.glossary[key].pop("parent_node", None)

# print(len(glossary_info.glossary.keys()))
# terms_to_remove = ['Day', 'Month', 'Year']
# glossary_info.glossary = {key: value for key, value in glossary_info.glossary.items() if value["term_name"] not in terms_to_remove}
# print(len(glossary_info.glossary.keys()))

In [ ]:
terms = []
for key, value in glossary_info.glossary.items():
    terms.append(value["term_name"])
print(len(terms))
print(terms)

In [35]:
# Function to extract threshold value from directory name
def get_threshold_from_dir(dir_name):
    return dir_name.split("_")[-1]


# Function to process a run directory and collect data for all thresholds
def process_run(run_path, column_filter):
    threshold_dirs = [
        d
        for d in os.listdir(run_path)
        if os.path.isdir(os.path.join(run_path, d)) and d.startswith("threshold_")
    ]
    run_data = {}

    for threshold_dir in threshold_dirs:
        threshold_value = get_threshold_from_dir(threshold_dir)
        csv_file = f"final_stats_threshold_{threshold_value}.csv"
        csv_path = os.path.join(run_path, threshold_dir, csv_file)

        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path, index_col=0)
            if column_filter in df.columns:
                # Extract required metrics
                run_data[threshold_value] = df[column_filter]

    return run_data


# Main function to collect data from all experiments and runs
def collect_data(main_dir, column_filter):
    # Dictionary to hold the data, outer dict for subdirectory, inner dict for threshold data
    all_data = {}

    sub_dirs = [
        d for d in os.listdir(main_dir) if os.path.isdir(os.path.join(main_dir, d))
    ]
    for sub_dir in sub_dirs:
        sub_dir_path = os.path.join(main_dir, sub_dir)
        run_dirs = [
            d
            for d in os.listdir(sub_dir_path)
            if os.path.isdir(os.path.join(sub_dir_path, d))
        ]
        for run_dir in run_dirs:
            run_path = os.path.join(sub_dir_path, run_dir)
            run_data = process_run(run_path, column_filter)

            if run_data:
                # Store the data by subdirectory, run, and threshold
                if sub_dir not in all_data:
                    all_data[sub_dir] = {}
                all_data[sub_dir][run_dir] = run_data

    return all_data


# Convert collected data into a DataFrame with multi-level columns
def create_dataframe_and_save(all_data, main_dir):
    dataframe_dict = {}
    for sub_dir, run_data in all_data.items():
        # Initialize an empty DataFrame to store the final combined results
        df_combined = pd.DataFrame()
        threshold_list = []
        counter = 1
        for run, thresholds in run_data.items():
            for threshold, metrics in thresholds.items():
                # Create multi-level column with format (threshold, run)
                column_name = (threshold, f"run_{counter}")
                df_combined[column_name] = metrics
                threshold_list.append(str(threshold))
            counter += 1

        # # For each experiment (sub_dir), calculate average across all runs for each threshold
        for threshold in set(threshold_list):
            threshold_cols = [col for col in df_combined.columns if threshold in col]
            avg_column_name = (threshold, f"run_average")
            df_combined[avg_column_name] = df_combined[threshold_cols].mean(axis=1)

        # Sort the columns for better readability
        df_combined = df_combined.sort_index(axis=1, level=[0, 1])
        save_to_csv(df_combined, main_dir / f"{sub_dir}.csv")
        dataframe_dict[sub_dir] = df_combined
    return dataframe_dict


# Save the DataFrame to a CSV
def save_to_csv(df, output_file="aggregated_results.csv"):
    df.to_csv(output_file)

In [36]:
from tqdm import tqdm


def write_experiment_details():
    with open(BASE_DIRECTORY / "experiment_details.txt", "w") as f:
        f.write(
            f"EXAMPLES_IN_GLOSSARY_TERMS: {EXAMPLES_IN_GLOSSARY_TERMS}\n"
            f"USE_COLUMN_INFO_WITH_DESCRIPTIONS: {USE_COLUMN_INFO_WITH_DESCRIPTIONS}\n"
            f"USE_TERM_DESCRIPTIONS: {USE_TERM_DESCRIPTIONS}\n"
            f"OLD_LABELS: {OLD_LABELS}"
        )
        f.close()
    # with open(BASE_DIRECTORY / "glossary_info.pkl", "wb") as f:
    #     pickle.dump(glossary_info, f)
    # with open(BASE_DIRECTORY / "tables_info.pkl", "wb") as f:
    #     pickle.dump(tables_info_dict, f)
    # with open(BASE_DIRECTORY / "columns_info.pkl", "wb") as f:
    #     pickle.dump(columns_info_dict, f)
    return None

In [ ]:
for experiment in tqdm(experiment_dict, total=len(experiment_dict)):
    (BASE_DIRECTORY / f"{experiment.experiment_name}").mkdir(
        parents=True, exist_ok=True
    )
    for run in range(experiment.iterations):
        print(f"run_{run}")
        # Create Output Directory:
        OUTPUT_DIR = (
            BASE_DIRECTORY
            / f"{experiment.experiment_name}"
            / f"run{run+1}_{datetime.now().strftime('%H-%M-%S')}"
        )
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        # write experiment details:
        write_experiment_details()

        # read and save prompt:
        prompt_path = current_dir / experiment.prompt_path
        prompt = prompt_path.read_text()
        pathlib.Path(OUTPUT_DIR / "prompt.txt").write_text(prompt)

        # Generate Response:
        parsed_llm_responses, raw_llm_responses = (
            get_parsed_responses_for_single_experiment_run(
                urns_dict=URNS_DICT,
                glossary_info=glossary_info,
                prompt_path=prompt_path,
            )
        )

        # Write response to directory
        write_llm_output_to_csv(
            llm_response=parsed_llm_responses,
            csv_path=OUTPUT_DIR / "parsed_response.csv",
        )
        convert_parsed_response_to_readable_csv(
            parsed_responses=parsed_llm_responses,
            columns_info_dict=columns_info_dict,
            desination_csv_path=OUTPUT_DIR / "parsed_response_with_high_confidence.csv",
            threshold=9,
        )
        with open(OUTPUT_DIR / "parsed_response.pkl", "wb") as fp:
            pickle.dump(parsed_llm_responses, fp)

        # Populate Analysis
        populate_analysis_for_confidence_threshold_list(
            output_dir=OUTPUT_DIR,
            confidence_thresholds=experiment.confidence_thresholds,
            filters=FILTERS,
            urns_dict=URNS_DICT,
            parsed_llm_responses=parsed_llm_responses,
            columns_info_dict=columns_info_dict,
            column_labels_csv_path=COLUMN_LABELS_CSV_PATH,
            label_column=LABEL_COLUMN,
        )

In [39]:
# Directory containing subdirectories of experiments
main_dir = BASE_DIRECTORY
column_filter = "no_filter"
aggregated_results_data = collect_data(main_dir, column_filter)
df_dict = create_dataframe_and_save(aggregated_results_data, main_dir)

In [ ]:
# SKIP: Only for testing purposes.
# LABEL_COLUMN = "glossary_term"
# populate_analysis_for_confidence_threshold_list(
#     output_dir=OUTPUT_DIR,
#     confidence_thresholds=experiment.confidence_thresholds,
#     filters=FILTERS,
#     urns_dict=URNS_DICT,
#     parsed_llm_responses=parsed_llm_responses,
#     columns_info_dict=columns_info_dict,
# )

In [ ]:
print("Done")

     **actual**            **predicted**           **cateory_name**
       None                    None                  match-no_assignment              TN
       None                    term                  mismatch-predicted_only          FP1
       term                    different term        mismatch                         FP2
       term                    term                  match-term_assigned              TP
       term                    None                  mismatch-actual_only             FN

    Recall of positive class:  TP/(TP + FN + FP2)
    Precision : TP/(TP + FP1 + FP2)

    Recal of negative class: TN/(TN + FP1)
    Precision of negative class: TN/(TN + FN)
